# AutoSelect 模块：自动方法选择

ScpTensor 的 AutoSelect 模块提供智能化的分析方法自动选择功能。该模块能够自动评估多种分析方法，并根据数据特征选择最优的方法组合。

## 模块简介

在单细胞蛋白质组学数据分析中，每个分析步骤都有多种可选方法：

- **标准化 (Normalization)**: 对数转换、中位数标准化、分位数标准化等
- **填充 (Imputation)**: KNN、MissForest、PPCA、SVD 等
- **批次校正 (Integration)**: ComBat、Harmony、MNN、Scanorama 等
- **降维 (Dimensionality Reduction)**: PCA、UMAP、t-SNE 等
- **聚类 (Clustering)**: K-means、Leiden、Louvain 等

AutoSelect 模块通过自动评估这些方法的性能，帮助您选择最适合当前数据的方法，无需手动调参。

## 核心优势

1. **自动化选择**: 无需手动测试各种方法组合
2. **数据驱动**: 根据实际数据特征进行评估
3. **可解释性**: 提供详细的评估报告和选择理由
4. **灵活性**: 支持自定义权重和方法列表
5. **流水线支持**: 支持多阶段自动选择

## 环境准备

首先导入所需的库和模块。

In [1]:
# 导入标准库
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

# 导入 AutoSelect 模块
import scptensor.autoselect as auto

# 导入 ScpTensor 核心模块
from scptensor import Assay, ScpContainer, ScpMatrix

# 设置 matplotlib 显示参数
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

print(f"ScpTensor AutoSelect 模块版本: {auto.__name__}")
print(f"支持的分析阶段: {auto.AutoSelector.SUPPORTED_STAGES}")

ScpTensor AutoSelect 模块版本: scptensor.autoselect
支持的分析阶段: ['normalize', 'impute', 'integrate', 'reduce', 'cluster']


## 加载示例数据

我们使用 T-SCP 数据集进行演示。

In [2]:
# 加载 T-SCP 数据集
from pathlib import Path

data_path = Path("../data/applications/cell_cycle/")

# 检查数据文件是否存在
data_file = data_path / "T-SCP_data.csv"
cells_file = data_path / "T-SCP_cells.csv"

if data_file.exists() and cells_file.exists():
    print("数据文件已找到:")
    print(f"  - 数据矩阵: {data_file}")
    print(f"  - 细胞信息: {cells_file}")
else:
    print("数据文件未找到，将创建模拟数据进行演示")

数据文件已找到:
  - 数据矩阵: ../data/applications/cell_cycle/T-SCP_data.csv
  - 细胞信息: ../data/applications/cell_cycle/T-SCP_cells.csv


In [ ]:
# 加载 T-SCP 数据并创建容器
# 加载表达矩阵 (第一列是样本ID, 第一行是蛋白质ID)
data_df = pl.read_csv(data_file)
meta_df = pl.read_csv(cells_file)

print(f"表达矩阵维度: {data_df.shape}")
print(f"样本元数据维度: {meta_df.shape}")

# 提取样本ID和蛋白质ID
protein_ids = data_df.columns[1:]  # 跳过第一列(样本ID列)
sample_ids = data_df[data_df.columns[0]].to_list()  # 第一列是样本ID

# 提取表达矩阵 (跳过第一列样本ID)
X = data_df.select(data_df.columns[1:]).to_numpy().astype(np.float64)

# 将0值替换为NaN (在单细胞蛋白质组学中,0通常表示缺失)
X[X == 0] = np.nan

print(f"\n表达矩阵形状: {X.shape} (样本数 x 蛋白质数)")
print(f"缺失值数量: {np.isnan(X).sum()}")
print(f"缺失值比例: {np.isnan(X).sum() / X.size * 100:.2f}%")

# 创建样本元数据 DataFrame
obs = pl.DataFrame({"_index": sample_ids})

# 从 meta_df 添加细胞周期信息
meta_index = meta_df[meta_df.columns[0]].to_list()
meta_cycle = meta_df["cell_cycle"].to_list()
cycle_dict = dict(zip(meta_index, meta_cycle, strict=False))

obs = obs.with_columns(
    pl.col("_index").replace_strict(cycle_dict, default="Unknown").alias("cell_cycle")
)

# 创建特征(蛋白质)元数据 DataFrame
var = pl.DataFrame({"_index": protein_ids})

# 创建 Mask 矩阵（记录缺失值）
m_matrix = np.zeros_like(X, dtype=np.int8)
m_matrix[np.isnan(X)] = 2  # 2 = LOD (below detection limit)

# 创建 ScpMatrix
matrix = ScpMatrix(X=X, M=m_matrix)

# 创建 Assay
assay = Assay(var=var, layers={"raw": matrix}, feature_id_col="_index")

# 创建 ScpContainer
container = ScpContainer(obs=obs, assays={"proteins": assay}, sample_id_col="_index")

print("\n数据容器创建成功!")
print(f"  - 样本数量: {container.n_samples}")
print(f"  - 蛋白质数量: {container.assays['proteins'].n_features}")
print(f"  - 可用层: {list(container.assays['proteins'].layers.keys())}")
print(f"  - 细胞周期阶段: {container.obs['cell_cycle'].unique().to_list()}")

---

## 第一章：单阶段自动选择

AutoSelect 模块提供了一系列便捷函数，用于单个分析阶段的自动方法选择。

### 1.1 auto_normalize() - 自动标准化选择

自动评估并选择最优的标准化方法。

**重要提示：标准化流程**

在蛋白质组学数据分析中，正确的处理顺序是：

```
原始数据 → 对数转换 (log_transform) → 标准化 (normalization)
```

`auto_normalize()` 函数会在对数转换后的数据上自动选择最优的标准化方法（中位数、均值或分位数标准化）。

In [ ]:
# 步骤 1: 首先应用对数转换（预处理）
from scptensor.normalization import log_transform

container_log = log_transform(
    container,
    assay_name="proteins",
    source_layer="raw",
    new_layer_name="log",
    base=2.0,
    offset=1.0,
)

print("对数转换完成！")
print(f"可用层: {list(container_log.assays['proteins'].layers.keys())}")

In [ ]:
# 步骤 2: 使用 auto_normalize 自动选择标准化方法
# 注意：这里应该在对数转换后的数据上进行
container_norm, norm_report = auto.auto_normalize(
    container=container_log,  # 使用对数转换后的数据
    assay_name="proteins",
    source_layer="log",  # 输入是对数转换后的层
    keep_all=False,  # 只保留最佳方法的结果
)

# 查看选择报告
print("=" * 60)
print("标准化方法选择报告")
print("=" * 60)
print(f"\n最佳方法: {norm_report.best_method}")
print(f"推荐理由: {norm_report.recommendation_reason}")

if norm_report.best_result:
    print("\n最佳方法评估指标:")
    for metric, score in norm_report.best_result.scores.items():
        print(f"  - {metric}: {score:.4f}")
    print(f"\n综合得分: {norm_report.best_result.overall_score:.4f}")
    print(f"执行时间: {norm_report.best_result.execution_time:.2f} 秒")

In [ ]:
# 查看所有测试方法的结果
print("\n所有测试方法结果:")
print("-" * 60)
for result in norm_report.results:
    status = "成功" if result.error is None else f"失败: {result.error}"
    best_marker = " (最佳)" if result == norm_report.best_result else ""
    print(f"{result.method_name}{best_marker}:")
    print(f"  - 综合得分: {result.overall_score:.4f}")
    print(f"  - 执行时间: {result.execution_time:.2f}s")
    print(f"  - 状态: {status}")

### 1.2 auto_impute() - 自动填充选择

自动评估并选择最优的缺失值填充方法。

In [ ]:
# 使用 auto_impute 自动选择填充方法
# 注意：填充通常在标准化之后进行
container_imputed, impute_report = auto.auto_impute(
    container=container_norm,
    assay_name="proteins",
    source_layer=norm_report.best_result.layer_name,  # 使用标准化后的数据
    keep_all=False,
)

# 查看选择报告
print("=" * 60)
print("填充方法选择报告")
print("=" * 60)
print(f"\n最佳方法: {impute_report.best_method}")
print(f"推荐理由: {impute_report.recommendation_reason}")

if impute_report.best_result:
    print("\n最佳方法评估指标:")
    for metric, score in impute_report.best_result.scores.items():
        print(f"  - {metric}: {score:.4f}")
    print(f"\n综合得分: {impute_report.best_result.overall_score:.4f}")

### 1.3 查看数据变化

让我们查看经过标准化和填充后数据的变化。

In [ ]:
# 比较原始数据和处理后数据
raw_data = container.assays["proteins"].layers["raw"].X
processed_data = container_imputed.assays["proteins"].layers[impute_report.best_result.layer_name].X

print("数据统计对比:")
print("-" * 40)
print("原始数据:")
print(f"  - 均值: {np.mean(raw_data[raw_data > 0]):.2f}")
print(f"  - 标准差: {np.std(raw_data[raw_data > 0]):.2f}")
print(f"  - 零值比例: {np.mean(raw_data == 0) * 100:.1f}%")

print("\n处理后数据:")
print(f"  - 均值: {np.mean(processed_data):.2f}")
print(f"  - 标准差: {np.std(processed_data):.2f}")
print(f"  - 零值比例: {np.mean(processed_data == 0) * 100:.1f}%")

---

## 第二章：完整流水线自动选择

对于完整的分析流程，可以使用 `AutoSelector` 类进行多阶段的自动选择。

### 2.1 AutoSelector 类介绍

`AutoSelector` 是自动选择系统的核心类，支持多阶段流水线分析。

In [ ]:
# 查看 AutoSelector 的配置选项
print("AutoSelector 支持的分析阶段:")
for stage in auto.AutoSelector.SUPPORTED_STAGES:
    print(f"  - {stage}")

print("\nAutoSelector 初始化参数:")
print("  - stages: 要执行的分析阶段列表（None = 全部）")
print("  - keep_all: 是否保留所有方法结果（False = 只保留最佳）")
print("  - weights: 自定义评估指标权重")
print("  - parallel: 是否并行执行（未来功能）")
print("  - n_jobs: 并行任务数量")

### 2.2 多阶段流水线设置

创建一个包含标准化、填充、降维和聚类的完整流水线。

In [ ]:
# 创建多阶段自动选择器
selector = auto.AutoSelector(
    stages=["normalize", "impute", "reduce", "cluster"],
    keep_all=False,  # 每个阶段只保留最佳方法
    parallel=True,
    n_jobs=-1,  # 使用所有可用 CPU 核心
)

print("已创建 AutoSelector 实例")
print(f"配置的阶段: {selector.stages}")
print(f"保留所有结果: {selector.keep_all}")

### 2.3 执行和结果分析

运行完整的自动选择流水线。

In [ ]:
# 执行自动选择流水线
# 注意：这可能需要一些时间
print("开始执行自动选择流水线...")

result_container, full_report = selector.run(
    container=container, assay_name="proteins", initial_layer="raw"
)

print("\n流水线执行完成!")

In [ ]:
# 打印完整报告摘要
print(full_report.summary())

In [ ]:
# 查看每个阶段的最佳方法
print("\n各阶段最佳方法汇总:")
print("=" * 50)
for stage_name, stage_report in full_report.stages.items():
    print(f"\n{stage_name.upper()}:")
    print(f"  最佳方法: {stage_report.best_method}")
    if stage_report.best_result:
        print(f"  综合得分: {stage_report.best_result.overall_score:.4f}")
        print(f"  结果层名: {stage_report.best_result.layer_name}")

---

## 第三章：自定义权重配置

AutoSelect 模块允许用户自定义评估指标的权重，以影响方法选择结果。

### 3.1 理解评估指标

每个分析阶段使用不同的评估指标：

In [ ]:
# 查看各阶段的默认评估指标权重
from scptensor.autoselect.evaluators.clustering import ClusteringEvaluator
from scptensor.autoselect.evaluators.imputation import ImputationEvaluator
from scptensor.autoselect.evaluators.normalization import NormalizationEvaluator

print("各阶段默认评估指标权重:")
print("=" * 50)

# 标准化
norm_eval = NormalizationEvaluator()
print("\n标准化 (Normalization):")
for metric, weight in norm_eval.metric_weights.items():
    print(f"  - {metric}: {weight}")

# 填充
impute_eval = ImputationEvaluator()
print("\n填充 (Imputation):")
for metric, weight in impute_eval.metric_weights.items():
    print(f"  - {metric}: {weight}")

# 聚类
cluster_eval = ClusteringEvaluator()
print("\n聚类 (Clustering):")
for metric, weight in cluster_eval.metric_weights.items():
    print(f"  - {metric}: {weight}")

### 3.2 设置自定义权重

创建一个更重视方差稳定性的标准化评估器。

In [ ]:
# 使用自定义权重创建选择器
custom_selector = auto.AutoSelector(
    stages=["normalize"],
    keep_all=True,  # 保留所有方法结果以便比较
    weights={
        "normalize": {
            "variance_stabilization": 0.6,  # 更重视方差稳定性
            "batch_effect": 0.2,
            "completeness": 0.2,
        }
    },
)

print("自定义权重配置:")
print(f"  标准化阶段权重: {custom_selector.weights.get('normalize', 'default')}")

In [ ]:
# 使用自定义权重运行
custom_container, custom_report = custom_selector.run(
    container=container, assay_name="proteins", initial_layer="raw"
)

print("使用自定义权重的选择结果:")
print(f"最佳方法: {custom_report.stages['normalization'].best_method}")
print(f"综合得分: {custom_report.stages['normalization'].best_result.overall_score:.4f}")

### 3.3 影响方法选择

不同的权重配置可能导致不同的方法选择结果。

In [ ]:
# 比较默认权重和自定义权重的结果
print("权重配置对方法选择的影响:")
print("-" * 50)

# 默认权重结果
default_stage = full_report.stages.get("normalization")
if default_stage:
    print("\n默认权重:")
    print(f"  最佳方法: {default_stage.best_method}")
    print(f"  得分: {default_stage.best_result.overall_score:.4f}")

# 自定义权重结果
custom_stage = custom_report.stages["normalization"]
print("\n自定义权重 (强调方差稳定性):")
print(f"  最佳方法: {custom_stage.best_method}")
print(f"  得分: {custom_stage.best_result.overall_score:.4f}")

---

## 第四章：保留所有方法结果

通过设置 `keep_all=True`，可以保留所有测试方法的结果，便于后续比较。

### 4.1 keep_all 参数

In [ ]:
# 创建保留所有结果的选择器
compare_selector = auto.AutoSelector(
    stages=["normalize"],
    keep_all=True,  # 保留所有方法结果
)

# 运行选择
compare_container, compare_report = compare_selector.run(
    container=container, assay_name="proteins", initial_layer="raw"
)

print("保留所有方法结果")
print(f"容器中的数据层: {list(compare_container.assays['proteins'].layers.keys())}")

### 4.2 比较不同方法的结果

In [ ]:
# 可视化比较不同方法的得分
from matplotlib.patches import Patch

stage_report = compare_report.stages["normalization"]

# 提取方法名和得分
methods = [r.method_name for r in stage_report.results]
scores = [r.overall_score for r in stage_report.results]
colors = ["green" if r == stage_report.best_result else "steelblue" for r in stage_report.results]

# 绘制条形图
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(methods, scores, color=colors)

# 添加标签
ax.set_xlabel("方法名称")
ax.set_ylabel("综合得分")
ax.set_title("标准化方法比较")
ax.set_ylim(0, 1)

# 添加得分标签
for bar, score in zip(bars, scores, strict=False):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{score:.3f}",
        ha="center",
        va="bottom",
    )

# 添加图例
legend_elements = [
    Patch(facecolor="green", label="最佳方法"),
    Patch(facecolor="steelblue", label="其他方法"),
]
ax.legend(handles=legend_elements, loc="upper right")

plt.tight_layout()
plt.savefig("normalization_comparison.png", dpi=300)
plt.show()

print("\n图表已保存为 normalization_comparison.png")

---

## 第五章：报告分析

AutoSelect 模块生成详细的报告，可以保存为多种格式。

### 5.1 AutoSelectReport 结构

In [ ]:
# 探索报告结构
print("AutoSelectReport 结构:")
print("=" * 50)

print("\n报告属性:")
print("  - stages: 包含各阶段报告的字典")
print(f"  - total_time: 总执行时间 ({full_report.total_time:.2f} 秒)")
print(f"  - warnings: 警告信息列表 ({len(full_report.warnings)} 条)")

print("\n已完成的阶段:")
for stage_name in full_report.stages:
    print(f"  - {stage_name}")

### 5.2 StageReport 详细信息

In [ ]:
# 查看单个阶段报告的详细信息
stage_name = "normalization"
if stage_name in full_report.stages:
    stage = full_report.stages[stage_name]

    print(f"{stage_name.upper()} 阶段报告:")
    print("-" * 40)
    print(f"阶段名称: {stage.stage_name}")
    print(f"测试方法数: {len(stage.results)}")
    print(f"成功率: {stage.success_rate:.1%}")
    print(f"最佳方法: {stage.best_method}")
    print(f"推荐理由: {stage.recommendation_reason}")

### 5.3 可视化评估结果

In [ ]:
# 创建评估结果热图
def plot_metrics_heatmap(stage_report, title="评估指标热图"):
    """绘制评估指标热图"""
    # 提取指标数据
    results = stage_report.results
    methods = [r.method_name for r in results]

    # 获取所有指标名称
    all_metrics = set()
    for r in results:
        all_metrics.update(r.scores.keys())
    metrics = sorted(all_metrics)

    # 创建数据矩阵
    data = np.zeros((len(methods), len(metrics)))
    for i, r in enumerate(results):
        for j, m in enumerate(metrics):
            data[i, j] = r.scores.get(m, 0.0)

    # 绘制热图
    fig, ax = plt.subplots(figsize=(10, max(5, len(methods) * 0.8)))

    im = ax.imshow(data, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)

    # 设置坐标轴
    ax.set_xticks(range(len(metrics)))
    ax.set_xticklabels(metrics, rotation=45, ha="right")
    ax.set_yticks(range(len(methods)))
    ax.set_yticklabels(methods)

    # 添加数值标签
    for i in range(len(methods)):
        for j in range(len(metrics)):
            text_color = "white" if data[i, j] < 0.5 else "black"
            ax.text(
                j, i, f"{data[i, j]:.2f}", ha="center", va="center", color=text_color, fontsize=9
            )

    # 添加颜色条
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("得分")

    ax.set_title(title)
    plt.tight_layout()
    return fig


# 绘制标准化阶段的指标热图
if "normalization" in full_report.stages:
    fig = plot_metrics_heatmap(full_report.stages["normalization"], title="标准化方法评估指标")
    plt.savefig("normalization_metrics_heatmap.png", dpi=300)
    plt.show()
    print("热图已保存为 normalization_metrics_heatmap.png")

### 5.4 保存报告

In [ ]:
# 保存报告为不同格式
output_dir = Path("./autoselect_reports")
output_dir.mkdir(exist_ok=True)

# 保存为 Markdown 格式
full_report.save(output_dir / "report.md", format="markdown")
print(f"已保存 Markdown 报告: {output_dir / 'report.md'}")

# 保存为 JSON 格式
full_report.save(output_dir / "report.json", format="json")
print(f"已保存 JSON 报告: {output_dir / 'report.json'}")

# 保存为 CSV 格式
full_report.save(output_dir / "report.csv", format="csv")
print(f"已保存 CSV 报告: {output_dir / 'report.csv'}")

In [ ]:
# 显示 Markdown 报告内容示例
report_path = output_dir / "report.md"
if report_path.exists():
    print("Markdown 报告预览:")
    print("-" * 50)
    with open(report_path) as f:
        content = f.read()
        # 显示前 50 行
        lines = content.split("\n")[:50]
        print("\n".join(lines))
        if len(content.split("\n")) > 50:
            print("\n... (更多内容请查看完整文件)")

---

## 第六章：高级用法

探索 AutoSelect 模块的高级功能。

### 6.1 单独使用评估器

In [ ]:
# 直接使用评估器进行更细粒度的控制
from scptensor.autoselect.evaluators.normalization import NormalizationEvaluator

# 创建评估器实例
evaluator = NormalizationEvaluator()

print("标准化评估器信息:")
print(f"阶段名称: {evaluator.stage_name}")
print(f"可用方法: {list(evaluator.methods.keys())}")
print(f"评估指标权重: {evaluator.metric_weights}")

In [ ]:
# 使用评估器运行所有方法
eval_container, eval_report = evaluator.run_all(
    container=container, assay_name="proteins", source_layer="raw", keep_all=True
)

print(f"评估完成，测试了 {len(eval_report.results)} 种方法")
print(f"最佳方法: {eval_report.best_method}")

In [ ]:
# 评估单个方法
method_name = "log_transform"
method_func = evaluator.methods[method_name]

single_container, single_result = evaluator.evaluate_method(
    container=container,
    method_name=method_name,
    method_func=method_func,
    assay_name="proteins",
    source_layer="raw",
)

print(f"单个方法评估结果 ({method_name}):")
print(f"  综合得分: {single_result.overall_score:.4f}")
print(f"  执行时间: {single_result.execution_time:.4f} 秒")
print(f"  错误信息: {single_result.error or '无'}")

### 6.2 自定义方法列表

In [ ]:
# 继承评估器以自定义方法列表
class CustomNormalizationEvaluator(NormalizationEvaluator):
    """自定义标准化评估器，只测试特定方法"""

    @property
    def methods(self):
        # 只返回特定的方法
        all_methods = super().methods
        # 可以在这里过滤或添加自定义方法
        return {k: v for k, v in all_methods.items() if "log" in k}


# 使用自定义评估器
custom_eval = CustomNormalizationEvaluator()
print(f"自定义评估器可用方法: {list(custom_eval.methods.keys())}")

### 6.3 并行执行

In [ ]:
# 配置并行执行
parallel_selector = auto.AutoSelector(
    stages=["normalize", "impute"],
    parallel=True,
    n_jobs=-1,  # 使用所有可用核心
)

print("并行执行配置:")
print(f"  - parallel: {parallel_selector.parallel}")
print(f"  - n_jobs: {parallel_selector.n_jobs}")

# 注意：并行执行是未来功能，目前阶段按顺序执行

### 6.4 错误处理和容错

In [ ]:
# 查看错误处理
print("评估器具有内置的错误处理:")
print("-" * 40)

for stage_name, stage_report in full_report.stages.items():
    failed_methods = [r for r in stage_report.results if r.error is not None]
    if failed_methods:
        print(f"\n{stage_name} 阶段失败的方法:")
        for r in failed_methods:
            print(f"  - {r.method_name}: {r.error}")
    else:
        print(f"\n{stage_name} 阶段: 所有方法执行成功")

    print(f"成功率: {stage_report.success_rate:.1%}")

---

## 总结

### 最佳实践

1. **从简单开始**: 先使用便捷函数（`auto_normalize`, `auto_impute`）熟悉模块
2. **检查报告**: 始终查看选择报告，理解为什么某个方法被选中
3. **自定义权重**: 根据您的研究需求调整评估指标权重
4. **保留结果比较**: 使用 `keep_all=True` 比较不同方法的结果
5. **保存报告**: 将报告保存为文件，便于后续分析和记录
6. **遵循正确的处理顺序**: 原始数据 → 对数转换 → 标准化 → 填充

### 常见问题

**Q: 如何知道哪个方法最适合我的数据？**  
A: AutoSelect 模块会自动评估并选择最佳方法。查看 `StageReport` 中的 `recommendation_reason` 了解选择理由。

**Q: 标准化的正确顺序是什么？**  
A: 正确的顺序是：首先对原始数据应用对数转换（`log_transform`），然后在转换后的数据上应用标准化方法（`norm_median`、`norm_mean` 或 `norm_quantile`）。AutoSelect 的 `auto_normalize` 函数会自动在对数转换后的数据上选择最优的标准化方法。

**Q: 可以只测试特定的方法吗？**  
A: 可以通过继承评估器类并重写 `methods` 属性来实现。

**Q: 评估指标的含义是什么？**  
- **variance_stabilization**: 方差稳定性，衡量数据方差的均匀程度
- **batch_effect**: 批次效应，衡量批次间差异的减少程度
- **completeness**: 完整性，衡量数据的完整程度
- **silhouette**: 轮廓系数，衡量聚类质量
- **rmse**: 均方根误差，衡量填充精度

**Q: 如何处理失败的评估？**  
A: 评估器会自动处理错误，失败的方法会被标记并跳过。查看 `EvaluationResult.error` 了解错误原因。

In [ ]:
# 快速参考：AutoSelect 模块主要 API
print("=" * 60)
print("ScpTensor AutoSelect 模块 API 速查")
print("=" * 60)

print("\n主要类:")
print("  - AutoSelector: 多阶段自动选择器")
print("  - AutoSelectReport: 完整流水线报告")
print("  - StageReport: 单阶段报告")
print("  - EvaluationResult: 单方法评估结果")

print("\n便捷函数:")
print("  - auto_normalize(): 自动标准化选择")
print("  - auto_impute(): 自动填充选择")
print("  - auto_integrate(): 自动批次校正选择")
print("  - auto_reduce(): 自动降维选择")
print("  - auto_cluster(): 自动聚类选择")

print("\n支持的分析阶段:")
for stage in auto.AutoSelector.SUPPORTED_STAGES:
    print(f"  - {stage}")

print("\n" + "=" * 60)
print("教程结束！感谢使用 ScpTensor AutoSelect 模块。")
print("=" * 60)